# Model

In [ ]:
import torch.nn as nn
import torch
import torch.nn as nn
import torchvision.transforms.functional as T

In [ ]:
class DoubleConv(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(DoubleConv, self).__init__()
    self.conv = nn.Sequential(
    nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias = False),
    nn.BatchNorm2d(out_channels),
    nn.ReLU(inplace = True),
    nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias = False),
    nn.BatchNorm2d(out_channels),
    nn.ReLU(inplace=True),
    )


  def forward(self, x):
    # print(x.shape)
    x = self.conv(x)
    # print(x.shape)
    return x

In [ ]:
class UNET(nn.Module):
  def __init__(self, in_channels = 3, out_channels = 104, features = [64, 128, 256, 512]):
    super().__init__()
    self.ups = nn.ModuleList()
    self.downs = nn.ModuleList()
    self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

    #down part of UNET
    for feature in features:
      self.downs.append(DoubleConv(in_channels, feature))
      in_channels = feature


    #Up part
    for feature in reversed(features):
      self.ups.append(
          nn.ConvTranspose2d(
              feature*2,
              feature,
              2,
              2,
          )
      )

      self.ups.append(DoubleConv(feature*2, feature))


    self.bottleneck = DoubleConv(features[-1], features[-1]*2)

    self.final_conv = nn.Conv2d(features[0], out_channels, 1,1)

  def forward(self, x):
    skip_connections = []
    print("\noriginal image\n")
    print(x.shape)

    print("\nnow into down\n")
    for down in self.downs:
      x = down(x)
      skip_connections.append(x)
      x = self.pool(x)
      print(x.shape)

    x = self.bottleneck(x)
    print('\nin bottleneck\n')
    print(x.shape)
    print("\nup\n")
    skip_connections = skip_connections[::-1]

    for idx in range(0, len(self.ups), 2):
      print(x.shape)
      x = self.ups[idx](x)
      skip_connection = skip_connections[idx//2]

      if x.shape != skip_connection.shape:
        x = T.resize(x, size = skip_connection.shape[2:])
      concat_skip = torch.cat((skip_connection, x), dim = 1)
      x = self.ups[idx+1](concat_skip)

    print('\n output \n')
    x = self.final_conv(x)
    # print(x.shape)

    return x

In [ ]:
def test():
  x = torch.randn((3, 1, 1361, 161))
  model = UNET(in_channels = 1, out_channels=1)
  preds = model(x)
  print(preds.shape)
  preds.shape == x.shape

if __name__ == "__main__":
  test()


original image

torch.Size([3, 1, 1361, 161])

now into down

torch.Size([3, 64, 680, 80])
torch.Size([3, 128, 340, 40])
torch.Size([3, 256, 170, 20])
torch.Size([3, 512, 85, 10])

in bottleneck

torch.Size([3, 1024, 85, 10])

up

torch.Size([3, 1024, 85, 10])
torch.Size([3, 512, 170, 20])
torch.Size([3, 256, 340, 40])
torch.Size([3, 128, 680, 80])

 output 

torch.Size([3, 1, 1361, 161])


# Dataset

In [13]:
from datasets import load_dataset
dataset = load_dataset("EduardoPacheco/FoodSeg103",cache_dir="./data/")

Generating validation split: 100%|██████████| 2135/2135 [00:00<00:00, 7793.17 examples/s]


In [14]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 4983
    })
    validation: Dataset({
        features: ['image', 'label', 'classes_on_image', 'id'],
        num_rows: 2135
    })
})

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
import numpy as np

In [ ]:
class CarvanaDataset(Dataset):
  def __init__(self, image_dir, mask_dir, transform = None):
    self.image_dir = image_dir
    self.mask_dir = mask_dir
    self.transform = transform
    self.images = os.listdir(image_dir)

  def __len__(self):
    return len(self.images)

  def __get__item__(self, index):

    img_path = os.path.join(self.image_dir, self.images[index])
    mask_path = os.path.join(self.mask_dir, self.images[index]replace('.jpg', '_mask.gif'))

    img = np.array(Image.open(img_path).convert('RGB'))
    mask = np.array(Image.open(mask_path).convert('L'), dtype = np.float32)

    mask[mask== 255.0] = 1.0

    if self.transform is not None:
      augmentations = self.transform(image = img, mask = mask)
      img = augmentations['image']
      mask = augmentations['mask']

    return img, mask


# Utils

In [ ]:
from ast import Num
import torchvision
import torch
from dataset import CarvanaDataset
from torch.utils.data import DataLoader

def save_checkpoint(state, filename='my_checkpoint.pth.tar'):
  print('=> Saving checkpoint')
  torch.save(state, filename)

def load_checkpoint(checkpoint, model):
  print('=> Loading checkpoint')
  model.load_state_dict(checkpoint['state_dict'])






In [ ]:
def get_loaders(train_dir,
                train_maskdir,
                val_dir,
                val_maskdir,
                batch_size,
                train_transform,
                val_transform,
                num_workers=4,
                pin_memory=True):

  train_ds = CarvanaDataset(image_dir=train_dir,
                           mask_dir=train_maskdir,
                           transform=train_transform,
                           )

  train_loader = DataLoader(train_ds,
                            batch_size=batch_size,
                            num_workers=num_workers,
                            pin_memory=pin_memory,
                            shuffle=True,
                            )

  val_ds = CarvanaDataset(image_dir=val_dir,
                          mask_dir=val_maskdir,
                          transform=val_transform,
                          )

  val_loader = DataLoader(val_ds,
                          batch_size=batch_size,
                          num_workers=num_workers,
                          pin_memory=pin_memory,
                          shuffle=False,
                          )

  return train_loader, val_loader


In [ ]:
def check_accuracy(loader, model, device = 'cuda'):
  num_correct = 0
  num_pixels = 0
  dice_score = 0
  model.eval()

  with torch.no_grad():
    for x, y in loader:
      x = x.to(device)
      y = y.to(device).unsqueeze(1)
      preds = torch.sigmoid(model(x))
      preds = (preds>0.5).float()
      num_correct += (preds == y).sum()
      num_pixels += torch.numel(preds)
      dice_score += (2*(preds*y).sum())/((preds +y).sum() +1e - 8)

  print(f"Got {num_correct}/{num_pixels} with acc {num_correct/num_pixels*100:.2f}")
  print(f"Dice score: {dice_score/len(loader)}")
  model.train()

In [ ]:
def save_predictions_as_imgs(
    loader, model, folder="saved_images/", device="cuda"
):
  model.eval()
  for idex, (x, y) in enumerate(loader):
    x = x.to(device)
    with torch.no_grad():
      preds = torch.sigmoid(model(x))
      preds = (preds > 0.5).float()

    torchvision.utils.save_image(
        preds, f"{folder}/pred_{idex}.png"
    )
    torchvision.utils.save_image(y.unsqueeze(1), f"{folder}{idex}"
    )

# Training

In [ ]:
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
# from A.pytorch import ToTensorV2
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim

# from utils import(
#     load_checkpoints,
#     save_checkpoints,
#     get_loaders,
#     check_accuracy,
#     save_predictions_as_imgs,
# )

In [ ]:
lr = 1e-4
device = 'cuda' if torch.cuda .is_available() else 'cpu'
batch = 32
epochs = 100
workers = 2
img_height =
img_width =
pin_mem = True
load_model = False
train_img_dir =
train_mask_dir =
val_img_dir =
val_mask_dir =

In [ ]:
def train_fn(loaer, model, optimizer, loss_fn, scaler):
  loop == tqdm(loader)

  for batch_idx, (data, targets) in enumerate(loop):
    data = data.to(device=device)
    targets = target.float().unsqueeze(1).to(device=device)

    #forward
    with torch.cude.amp.autocast():
      predictions = model(data)
      loss = loss_fn(predictions, targets)

    #backward
    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    #update tqdm loop
    loop.set_postfix(loss = loss.item())

def main():
  train_transform = A.Compose([
      A.Resize(height = img_height, width = img_width),
      A.Rotate(limit = 35, p = 1.0),
      A.HorizontalFlip(p = 0.5),
      A.VerticalFlip(p = 0.1),
      A.Normalize(
          mean = [0.0, 0.0, 0.0],
          std = [1.0, 1.0, 1.0],
          max_pixel_value = 255.0,
      ),
      ToTensorV2(),
  ])

  val_transform = A.Compose([
      A.Resize(height = img_height, width = img_width),
      A.Normalize(
          mean = [0.0, 0.0, 0.0],
          std = [1.0, 1.0, 1.0]
          max_pixel_value = 255.0,
      ),
      ToTensorV2(),
  ])

  model = UNET(in_channels = 3, out_channels = 1).to(device)
  loss_fn = nn.BCEWithLogitsLoss()
  optimizer = optim.Adam(model.parameters(), lr = lr)

  train_loader, val_loader = get_loader(
      train_img_dir,
      train_mask_dir,
      val_img_dir,
      val_mask_dir,
      batch,
      train_transform,
      val_transform,
  )


  for epoch in range(epochs):
    train_fn(train_loader, model, optimizer, loss_fn, scaler)

    #save model

    checkpoint = {
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }
    save_checkpoint(checkpoint)


    #check_accuracy
    check_accuracy(val_loader, model, device = device)

    #print some examples
    save_predictions_as_imgs(val_loader, model, folder = 'saved_images/', device = device)

if __name__ == '__main__':
  main()